# CausalRank — 从原始数据到多 Treatment 因果估计（Phase 1）

本 Notebook 演示 **Phase 1** 的完整链路：

1. 已知真值的合成数据（含混杂）——`causalrank.simulation` 提供 `mu_true = E[Y(a)]`；
2. 原始组均值为什么是有偏的（混杂偏差）；
3. 多分类 GBM 倾向得分 + Hájek IPTW 得到同一目标总体上的 `mu_a`；
4. 重叠 / ESS / SMD 诊断与质量门控；
5. 弱重叠场景下**诚实的失败样貌**：宽警告而非“可靠排名”。

> **Phase 1 不提供统计推断**：`se / ci / ranks` 为空，联合 Bootstrap 协方差（Phase 2）
> 与排名置信集（Phase 3）尚未实现。请勿把点估计排序当作已确立的统计排名。
> 因果解释依赖一致性、无未观测混杂、positivity 与 iid 抽样等**无法用数据验证**的假设。

In [1]:
import pandas as pd

from causalrank import CausalRank
from causalrank.simulation import make_dataset, scenario_names

pd.set_option("display.width", 120)
print("available scenarios:", scenario_names())

available scenarios: ('randomized', 'confounded', 'strong_confounding', 'weak_overlap', 'rare_treatment', 'equal_effects', 'close_effects', 'mixed_missing')


## 1. 合成数据：已知真实潜在结果均值

`confounded` 场景：多项 softmax 分配 + 二元潜在结果。真值
$\mu_a = E_X[P(Y(a)=1\mid X)]$ 由**独立**的 20 万条 Monte Carlo 参考协变量样本估计，
参考 seed 与 MC 标准误都会记录下来。

In [2]:
ds = make_dataset("confounded", n=4000, random_state=2026)
print("reference: n =", ds.n_reference, ", seed =", ds.reference_seed)
truth = pd.DataFrame({"mu_true": ds.mu_true, "mc_se": ds.mu_true_mc_se})
display(truth.round(5))
ds.df.head()

reference: n = 200000 , seed = 20260101


,mu_true,mc_se
A,0.43670,0.00027
B,0.51845,0.00028
C,0.40239,0.00027


,x1,x2,x3,x4,treatment,outcome
0,-0.793122,1.496042,1.0,b,A,0.0
1,0.240571,-0.917138,0.0,c,C,0.0
2,-1.896326,0.023777,0.0,a,C,1.0
3,1.395772,-0.993125,0.0,a,B,1.0
4,0.638295,0.403208,0.0,c,A,0.0


## 2. 朴素对比：原始组均值为什么有偏

各 Treatment 组的协变量分布不同（混杂），所以原始组均值比较的是**不同的人群**，
不是同一目标总体上的 $E[Y(a)]$。

In [3]:
raw = ds.df.groupby("treatment")["outcome"].agg(["mean", "count"])
raw.columns = ["raw_mean", "n"]
raw["mu_true"] = ds.mu_true
raw["raw_bias"] = raw["raw_mean"] - raw["mu_true"]
display(raw.round(4))
print("mean |raw bias| =", round(float(raw["raw_bias"].abs().mean()), 4))

,raw_mean,n,mu_true,raw_bias
treatment,,,,
A,0.4920,1630,0.4367,0.0553
B,0.4585,1228,0.5185,-0.0600
C,0.4221,1142,0.4024,0.0197


mean |raw bias| = 0.045


## 3. 估计器：多分类 GBM 倾向得分 + Hájek IPTW

$$\hat e_a(x)=\widehat{P}(T=a\mid X=x),\qquad w_i=\frac{1}{\hat e_{T_i}(X_i)},\qquad
\hat\mu_a^{H}=\frac{\sum_i 1\{T_i=a\}w_iY_i}{\sum_i 1\{T_i=a\}w_i}$$

- 倾向得分是**预测模型**（联合多分类 GBM），IPTW 才是因果估计方法；
- 插补 / 编码 / 拟合全部包在 sklearn Pipeline 内，`cross_fit=5` 产生 out-of-fold 倾向得分，
  避免“同一份数据既拟合又评估”导致的过拟合权重；
- 所有 Treatment 都在**同一目标总体**上估计，排名目标才可比。

In [4]:
model = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    estimator="iptw",
    propensity_model="gbm",
    estimand="population_mean",
    weighting="hajek",
    cross_fit=5,
    confidence_level=0.95,
    random_state=2026,
)
result = model.fit(ds.df)
result.effects.to_dataframe().round(4)

,label,n_treated,raw_mean,mu,se,ci_low,ci_high,weight_sum,ess
0,A,1630,0.4920,0.4551,NaN,NaN,NaN,4174.7073,1374.1923
1,B,1228,0.4585,0.5218,NaN,NaN,NaN,4345.0186,714.2607
2,C,1142,0.4221,0.3927,NaN,NaN,NaN,4257.8338,949.3240


In [5]:
est = result.effects.to_dataframe().set_index("label")
compare = pd.DataFrame(
    {
        "mu_hat": est["mu"],
        "mu_true": ds.mu_true,
        "err_adjusted": est["mu"] - ds.mu_true,
        "raw_mean": est["raw_mean"],
        "err_raw": est["raw_mean"] - ds.mu_true,
    }
)
display(compare.round(4))
print("mean |err| adjusted =", round(float(compare["err_adjusted"].abs().mean()), 4))
print("mean |err| raw      =", round(float(compare["err_raw"].abs().mean()), 4))

,mu_hat,mu_true,err_adjusted,raw_mean,err_raw
A,0.4551,0.4367,0.0184,0.4920,0.0553
B,0.5218,0.5185,0.0033,0.4585,-0.0600
C,0.3927,0.4024,-0.0097,0.4221,0.0197


mean |err| adjusted = 0.0105
mean |err| raw      = 0.045


## 4. 诊断：权重、重叠、平衡与质量门控

- **ESS**：$\mathrm{ESS}_a=(\sum_i w_{ia})^2/\sum_i w_{ia}^2$，衡量权重把有效信息集中到多少条观测上；
- **overlap**：实际 Treatment 概率 $e_{T_i}(X_i)$ 的分布，极小值意味着 positivity 弱；
- **SMD**：加权前后协变量在 Treatment 两两之间的标准化均值差（分母用未加权 pooled SD）。

诊断只能**暴露经验问题**，不能验证“无未观测混杂”等识别假设。

In [6]:
print(result.diagnostics.summary())

=== CausalRank diagnostics (Phase 1: estimation only, no inference) ===

-- group summary (observed data) --
label    n  share  raw_outcome_mean  mu_hajek_iptw
    A 1630 0.4075          0.492025       0.455143
    B 1228 0.3070          0.458469       0.521773
    C 1142 0.2855          0.422067       0.392739

-- weight distribution (Hajek IPTW weights w = 1/e_{T}(X)) --
label    n  weight_sum         ess  ess_ratio    w_min     w_max  max_weight_share    w_p01    w_p05    w_p50    w_p95     w_p99
    A 1630 4174.707344 1374.192323   0.843063 1.251110 13.754780          0.003295 1.407876 1.579312 2.284235 4.492160  6.488381
    B 1228 4345.018617  714.260674   0.581646 1.111899 27.771883          0.006392 1.180923 1.332394 2.623514 8.955633 18.490573
    C 1142 4257.833781  949.324032   0.831282 1.607023 19.885860          0.004670 1.908368 2.176759 3.366041 6.290905 10.713499

-- overlap: observed-treatment propensity e_{T}(X) --
label  e_arm_min  e_arm_max  e_obs_min  e_obs_median 

In [7]:
for w in result.warnings:
    print(f"{w.level:>8} | {w.code:<28} | {w.message}")
print("\nquality gate:", result.diagnostics.gate)


quality gate: pass


## 5. 弱重叠场景：诚实的失败样貌

当 Treatment 分配接近确定性、共同支持不足时，IPTW 权重会爆炸、ESS 崩塌。
正确的行为是**给出强警告并降级质量门控**，而不是输出一个看起来精确的排名。

In [8]:
weak = make_dataset("weak_overlap", n=3000, random_state=7)
weak_result = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    cross_fit=5,
    random_state=2026,
).fit(weak.df)
display(weak_result.effects.to_dataframe().round(4))
print("quality gate:", weak_result.diagnostics.gate)
for w in weak_result.warnings:
    print(f"{w.level:>8} | {w.code:<22} | {w.message[:110]}")

,label,n_treated,raw_mean,mu,se,ci_low,ci_high,weight_sum,ess
0,A,1407,0.5338,0.5861,NaN,NaN,NaN,3411.2256,29.1795
1,B,1339,0.4332,0.3696,NaN,NaN,NaN,3304.5597,30.3912
2,C,254,0.4331,0.3846,NaN,NaN,NaN,5813.7488,30.9563


quality gate: fail
 warning | low_ess                | Treatment 'A': effective sample size ESS = 29.2 (< min_ess = 30.0). The weighted mean for this arm is dominate
 warning | low_ess                | Treatment 'A': ESS ratio = 0.021 (< min_ess_ratio = 0.2); extreme weights are concentrating the estimate.
 warning | extreme_weight         | Treatment 'A': the largest weight carries 17.4% of the arm's total weight (> 10%). Check overlap for this arm.
 warning | low_ess                | Treatment 'B': ESS ratio = 0.023 (< min_ess_ratio = 0.2); extreme weights are concentrating the estimate.
 warning | extreme_weight         | Treatment 'B': the largest weight carries 15.6% of the arm's total weight (> 10%). Check overlap for this arm.
 warning | low_ess                | Treatment 'C': ESS ratio = 0.122 (< min_ess_ratio = 0.2); extreme weights are concentrating the estimate.
 warning | extreme_weight         | Treatment 'C': the largest weight carries 11.6% of the arm's total weight (> 1

## 6. 本版本的边界

1. **无推断**：`se / ci / ranks` 为空（`result.inference_available is False`）；
   Phase 2 引入“每轮重抽样 + 重新拟合全部协议”的联合 Bootstrap 协方差，
   Phase 3 引入 Bonferroni 同时差值区间与保守排名置信集；
2. **不检验识别假设**：无未观测混杂、一致性、positivity 由使用者负责论证；
3. **不把 AUC 当证据**：倾向模型的分类表现不能替代 overlap / balance 诊断；
4. **iid 假设**：重复记录同一客户/家庭时行 Bootstrap 不成立（本版本只提示）；
5. **不做**：AIPW、Overlap Weighting、PSM、DML、个体层最优决策——明确列为后续研究。

详细定义见 `docs/methodology.md` 与 `docs/assumptions.md`。